[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/03_quality_filtering.ipynb)

# Step 3 — Quality Filtering and LLM-as-Judge

Filter **both** raw training corpora from Step 2: strategy Q&A (heuristics + judge) and instruction back-translation (heuristics). Save the two SFT files notebook 04 trains on.

## Learning objectives
- Apply deduplication, format checks, and prompt-leakage detection
- Score strategy samples on correctness, coherence, instruction-following, and plausibility
- Heuristic-filter the IBT corpus and save the final back-translation SFT file
- Optionally compare candidates pairwise against seed examples

In [1]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    DEFAULT_JUDGE_THRESHOLD,
    RESULTS_DIR,
    SYNTHETIC_FILTERED_PATH,
    SYNTHETIC_IBT_PATH,
    SYNTHETIC_IBT_RAW_PATH,
    SYNTHETIC_RAW_PATH,
    QASample,
    apply_heuristic_filters,
    create_judge_client,
    filter_with_judge,
    load_implementation_dotenv,
    load_typed_jsonl,
    save_typed_jsonl,
    summarize_heuristic_rejections,
    summarize_judge_scores,
    use_repo_root,
    write_json,
)
from rich import box
from rich.console import Console
from rich.table import Table


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_implementation_dotenv()
use_repo_root(Path("."))

console = Console(width=100)

Notebook path is already the root path: /Users/royajavadi/projects/synthetic-data-bootcamp


## 1. Heuristic filtering (strategy corpus)

Load `synthetic_raw.jsonl` from Step 2 and drop obvious junk before the judge.

In [3]:
raw_samples = load_typed_jsonl(SYNTHETIC_RAW_PATH, QASample.from_dict)
kept, rejected = apply_heuristic_filters(raw_samples)
print(f"Kept {len(kept)} / {len(raw_samples)} samples")

reason_counts = summarize_heuristic_rejections(rejected)
print("Rejection reasons:", reason_counts)

# Inspect a few rejected samples and which heuristic(s) fired
table = Table(title="Sample heuristic rejections", show_lines=True)
table.add_column("id", style="cyan", max_width=20)
table.add_column("reason(s)", style="red")
table.add_column("question", overflow="fold")
for row in rejected[:8]:
    table.add_row(row.get("id", ""), row.get("reasons", row.get("reason", "")), row.get("question", ""))
console.print(table)

Kept 135 / 144 samples
Rejection reasons: {'duplicate_question': 9}


                                    Sample heuristic rejections                                     
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ id                   ┃ reason(s)          ┃ question                                             ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ d9c3cd28-0bdb-47a1-… │ duplicate_question │ Under the 'PROMISE TO PAY' section, what are the     │
│                      │                    │ specific types of charges that you promise to pay if │
│                      │                    │ they are made to y                                   │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ f2ff0759-ce0a-44dd-… │ duplicate_question │ Under what specific condition can a Statement Copy   │
│                      │                    │ Fee be charged to an account, and what is the        │
│                      │                    │ explicit exception to t                              │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 5d307b20-977b-47ed-… │ duplicate_question │ Under what condition can a change made to the        │
│                      │                    │ Consumer Credit Card Agreement by the Credit Union   │
│                      │                    │ apply to your existing                               │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ c4c7450a-d7d8-45d8-… │ duplicate_question │ Under what specific conditions will the issuer close │
│                      │                    │ a primary account holder's account and require them  │
│                      │                    │ to apply for a                                       │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 4b692639-0b81-4a79-… │ duplicate_question │ Under the 'NO WAIVER' provision, what is the         │
│                      │                    │ consequence if the Credit Union repeatedly delays    │
│                      │                    │ enforcing its rights?                                │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 2e8be074-b274-447d-… │ duplicate_question │ If a customer suspects an error and reports it via a │
│                      │                    │ phone call, what are the two consequences regarding  │
│                      │                    │ the investigati                                      │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ c5704cbf-d453-400c-… │ duplicate_question │ According to the Updated Investor Bulletin from      │
│                      │                    │ April 23, 2026, which specific presidential          │
│                      │                    │ directive prompted the SEC's                         │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 9d8ecc06-d25f-4362-… │ duplicate_question │ According to the passage, what are the specific      │
│                      │                    │ criteria that a passphrase must meet to be           │
│                      │                    │ considered "strong," and what                        │
└──────────────────────┴────────────────────┴──────────────────────────────────────────────────────┘

## 2. LLM-as-judge absolute scoring

Here the judge model evaluates **synthetic Q&A quality** (question + gold answer vs the source passage). This is different from notebooks 01/04, where the judge scores a **model response** against a reference answer after inference.

The minimum pass score is defined in configs (`DEFAULT_JUDGE_THRESHOLD`).


In [4]:
from IPython.display import HTML
from IPython.display import display as ipy_display


judge = create_judge_client()

heuristic_rejected = rejected
state = {"done": 0}
total = len(kept)
handle = ipy_display(
    HTML(f"<b>Judge scoring</b> 0/{total} <progress value='0' max='{total}' style='width:24em'></progress>"),
    display_id=True,
)


def on_progress() -> None:
    """Refresh the in-notebook judge scoring progress bar."""
    state["done"] += 1
    done = state["done"]
    handle.update(
        HTML(
            f"<b>Judge scoring</b> {done}/{total} <progress value='{done}' max='{total}' style='width:24em'></progress>",
        ),
    )


filtered_samples, judge_scores, judge_rejected = filter_with_judge(
    judge,
    kept,
    threshold=DEFAULT_JUDGE_THRESHOLD,
    on_progress=on_progress,
)

# filter_with_judge re-runs heuristics on `kept`, so keep the original heuristic
# rejections and append judge-only rejects for an accurate quality report.
rejected = heuristic_rejected + [row for row in judge_rejected if row.get("reason") == "below_judge_threshold"]
console.print(
    f"[bold green]After judge filter:[/bold green] [yellow]{len(filtered_samples)}[/yellow] kept, "
    f"[red]{len(rejected)}[/red] rejected "
    f"[dim]({len(heuristic_rejected)} heuristic + "
    f"{len(rejected) - len(heuristic_rejected)} judge)[/dim]",
)
summarize_judge_scores(judge_scores)

2026-09-18 13:52:20,141 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 783137e2-cc78-4f6d-b8ff-44db910ec979
2026-09-18 13:52:23,321 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 261b8810-8a7c-455b-8412-d27148124d3c
2026-09-18 13:52:24,965 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 6a39150c-497c-4cf2-9ac5-c9f3c951a555
2026-09-18 13:52:26,405 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 64344d6a-d808-4885-833f-3979e423a8b3
2026-09-18 13:52:27,835 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 03cb991c-91b9-411e-886f-4853b5253f05
2026-09-18 13:52:28,894 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 524d3463-e606-448e-b7de-9b8800737973
2026-09-18 13:52:30,388 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 85f029d4-9f5d-4b19-842e-3c5bfb5c0995
2026-09-18 13:52:31,734 INFO aieng.syn_data.text.judge:

After judge filter: 135 kept, 9 rejected (9 heuristic + 0 judge)

{'correctness': 4.992592592592593,
 'coherence': 5.0,
 'instruction_following': 5.0,
 'factual_plausibility': 4.992592592592593,
 'average': 4.996296296296296}

## 3. Save filtered strategy corpus and quality report

In [6]:
save_typed_jsonl(
    SYNTHETIC_FILTERED_PATH,
    filtered_samples,
    to_dict=QASample.to_dict,
)

quality_report = {
    "input_count": len(raw_samples),
    "after_heuristics": len(kept),
    "after_judge": len(filtered_samples),
    "judge_threshold": DEFAULT_JUDGE_THRESHOLD,
    "judge_summary": summarize_judge_scores(judge_scores),
    "heuristic_rejected_count": len(heuristic_rejected),
    "judge_rejected_count": len(rejected) - len(heuristic_rejected),
    "rejected": rejected,
}
write_json(RESULTS_DIR / "quality_report.json", quality_report)


table = Table(title="Quality Report", box=box.ROUNDED)
table.add_column("Metric", style="bold cyan")
table.add_column("Value", style="bold yellow")

for k, v in quality_report.items():
    if isinstance(v, dict):
        # If value is a dictionary, show sub-keys and values
        for subk, subv in v.items():
            table.add_row(f"{k}.{subk}", str(subv))
    elif isinstance(v, list):
        table.add_row(k, f"{len(v)} items")
    else:
        table.add_row(k, str(v))
console.print(table)

                      Quality Report                       
╭─────────────────────────────────────┬───────────────────╮
│ Metric                              │ Value             │
├─────────────────────────────────────┼───────────────────┤
│ input_count                         │ 144               │
│ after_heuristics                    │ 135               │
│ after_judge                         │ 135               │
│ judge_threshold                     │ 3.5               │
│ judge_summary.correctness           │ 4.992592592592593 │
│ judge_summary.coherence             │ 5.0               │
│ judge_summary.instruction_following │ 5.0               │
│ judge_summary.factual_plausibility  │ 4.992592592592593 │
│ judge_summary.average               │ 4.996296296296296 │
│ heuristic_rejected_count            │ 9                 │
│ judge_rejected_count                │ 0                 │
│ rejected                            │ 9 items           │
╰─────────────────────────────────────┴───────────────────╯

## 4. Filter and save the IBT SFT corpus

Load the raw back-translation file from Step 2. IBT already has a generation-time overlap check, so this path uses **heuristics only** (same `apply_heuristic_filters` as section 1) — not the LLM judge. One sample per paragraph usually means few or no duplicate-question drops.

Save the kept rows to `synthetic_back_translation.jsonl` for notebook 04.

In [7]:
ibt_raw_samples = load_typed_jsonl(SYNTHETIC_IBT_RAW_PATH, QASample.from_dict)
ibt_final_samples, ibt_rejected = apply_heuristic_filters(ibt_raw_samples)
print(f"Final IBT SFT corpus size: {len(ibt_final_samples)} (rejected {len(ibt_rejected)} / {len(ibt_raw_samples)})")
print("IBT rejection reasons:", summarize_heuristic_rejections(ibt_rejected))

ibt_table = Table(title="Sample IBT heuristic rejections", show_lines=True)
ibt_table.add_column("id", style="cyan", max_width=20)
ibt_table.add_column("reason(s)", style="red")
ibt_table.add_column("question", overflow="fold")
for row in ibt_rejected[:8]:
    ibt_table.add_row(row.get("id", ""), row.get("reasons", row.get("reason", "")), row.get("question", ""))
if ibt_rejected:
    console.print(ibt_table)
else:
    console.print("No IBT heuristic rejections — expected when generating one sample per paragraph.")

save_typed_jsonl(
    SYNTHETIC_IBT_PATH,
    ibt_final_samples,
    to_dict=QASample.to_dict,
)
console.print(f"Saved {len(ibt_final_samples)} IBT SFT samples to {SYNTHETIC_IBT_PATH}")

Final IBT SFT corpus size: 36 (rejected 0 / 36)
IBT rejection reasons: {}


No IBT heuristic rejections — expected when generating one sample per paragraph.

Saved 36 IBT SFT samples to 
/Users/royajavadi/projects/synthetic-data-bootcamp/implementations/qa_text_generation/data/synthetic
/synthetic_back_translation.jsonl